# Семинар 1. Frozen Lake: политика, её оценка и метод Cross-Entropy

Семинар идёт по лекции: та же среда, те же функции, но ключевые куски вы пишете сами. Места для кода помечены `TODO`, после большинства из них стоит проверка через `assert` — если она прошла, можно двигаться дальше.

План:

1. Интерфейс Gymnasium на примере `FrozenLake-v1`
2. Политика как таблица: маршрут руками и что ломается на скользком льду
3. Оценка политики методом Монте-Карло и ценность клетки
4. Метод Cross-Entropy по шагам: отбор элиты, обновление политики, цикл
5. Эксперименты: сглаживание, скользкий лёд, поле 8×8
6. Что дальше

Первая ячейка — служебная, из лекции: функции для картинок. Запустить и не читать.

In [ ]:
# Служебный код для картинок: запустить и не читать. Содержательный код начинается со следующей ячейки.
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from matplotlib import animation
from IPython.display import HTML, display

ARROWS = "←↓→↑"                      # так пронумерованы действия в Frozen Lake: 0=←, 1=↓, 2=→, 3=↑
CELL_COLORS = {"S": "#f2f2f2", "F": "#ffffff", "H": "#a8c8ec", "G": "#a9dfa9"}

def lake_desc(env):
    """Карта озера как список строк: ['SFFF', 'FHFH', 'FFFH', 'HFFG']."""
    return ["".join(c.decode() for c in row) for row in env.unwrapped.desc]

def draw_lake(desc, ax=None, size=3.0):
    """Пустая карта озера. Возвращает оси, поверх которых можно рисовать маршруты и стрелки."""
    nrow, ncol = len(desc), len(desc[0])
    if ax is None:
        _, ax = plt.subplots(figsize=(size, size * nrow / ncol))
    ax.set_xlim(0, ncol); ax.set_ylim(nrow, 0); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    for r in range(nrow):
        for c in range(ncol):
            ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=CELL_COLORS[desc[r][c]], edgecolor="k", lw=0.6))
            if desc[r][c] in "SHG":
                ax.text(c + 0.5, r + 0.82, desc[r][c], ha="center", va="center", fontsize=8, color="#666")
    return ax

def draw_paths(sessions, desc, ax=None, title="", color="C0", alpha=0.35, size=3.0):
    """Маршруты эпизодов поверх карты: ломаная через центры клеток, точка — где эпизод закончился."""
    ax = draw_lake(desc, ax, size)
    ncol, rng = len(desc[0]), np.random.default_rng(0)
    for states, *_ in sessions:
        pts = np.array([divmod(int(s), ncol) for s in states], float) + 0.5 + rng.normal(0, 0.07, (len(states), 2))
        ax.plot(pts[:, 1], pts[:, 0], color=color, alpha=alpha, lw=1.5)
        ax.plot(pts[-1, 1], pts[-1, 0], "o", color=color, alpha=alpha, ms=4)
    ax.set_title(title, fontsize=10)
    return ax

def draw_policy(policy, desc, ax=None, title="", size=3.0):
    """Стрелка — самое вероятное действие, насыщенность — его вероятность; «·» — строка почти равномерная."""
    ax = draw_lake(desc, ax, size)
    ncol = len(desc[0])
    for s, p in enumerate(policy):
        r, c = divmod(s, ncol)
        if desc[r][c] in "HG":
            continue
        a, top2 = int(np.argmax(p)), np.sort(p)[-2:]
        if top2[1] - top2[0] < 0.05:
            ax.text(c + 0.5, r + 0.5, "·", ha="center", va="center", fontsize=18, color="grey")
        else:
            ax.text(c + 0.5, r + 0.5, ARROWS[a], ha="center", va="center", fontsize=20, alpha=0.2 + 0.8 * float(p[a]))
    ax.set_title(title, fontsize=10)
    return ax

def plot_values(values, desc, ax=None, title="", vmax=None, size=3.0):
    """Тепловая карта ценности клеток (проруби и цель — терминальные, у них ценность 0 по определению)."""
    nrow, ncol = len(desc), len(desc[0])
    ax = draw_lake(desc, ax, size)
    grid = np.asarray(values, float).reshape(nrow, ncol)
    vmax = vmax or max(float(np.nanmax(grid)), 1e-9)
    for r in range(nrow):
        for c in range(ncol):
            if desc[r][c] in "HG":
                continue
            if np.isnan(grid[r, c]):                     # клетку ни разу не посещали — оценки нет
                ax.text(c + 0.5, r + 0.5, "—", ha="center", va="center", fontsize=9, color="grey")
                continue
            ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=plt.cm.YlOrRd(0.85 * grid[r, c] / vmax), edgecolor="k", lw=0.6))
            ax.text(c + 0.5, r + 0.5, f"{grid[r, c]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(title, fontsize=10)
    return ax

def plot_returns(returns, threshold=None, elite_mask=None, ax=None, title=""):
    """Каждая точка — эпизод, по вертикали его return (как на схеме метода Cross-Entropy)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.6, 3.2))
    returns, x = np.asarray(returns, float), np.random.default_rng(0).uniform(0, 1, len(returns))
    if elite_mask is None:
        ax.scatter(x, returns, s=22, color="C0", alpha=0.7)
    else:
        m = np.asarray(elite_mask, bool)
        ax.scatter(x[~m], returns[~m], s=18, color="grey", alpha=0.35, label=f"остальные ({(~m).sum()})")
        ax.scatter(x[m], returns[m], s=42, color="C2", alpha=0.95, edgecolors="white", label=f"элита ({m.sum()})")
        ax.legend(loc="upper right", fontsize=8)
    if threshold is not None:
        ax.axhline(threshold, ls="--", color="C3", lw=1.5)
        ax.text(0.0, threshold + 0.02, f"порог = {threshold:.2f}", color="C3", fontsize=8)
    ax.set_xticks([]); ax.set_ylabel("return"); ax.set_ylim(-0.05, 1.05); ax.set_title(title, fontsize=10)
    return ax

def animate_policy(policies, rates, desc, interval=500):
    """Кадр — одна итерация обучения: стрелки политики и доля успешных эпизодов."""
    nrow, ncol = len(desc), len(desc[0])
    fig, ax = plt.subplots(figsize=(3.4, 3.4 * nrow / ncol + 0.3)); plt.close(fig)
    def frame(k):
        ax.clear(); draw_policy(policies[k], desc, ax=ax, title=f"итерация {k}: доля успехов {rates[k]:.0%}")
    anim = animation.FuncAnimation(fig, frame, frames=len(policies), interval=interval)
    return HTML(anim.to_jshtml(default_mode="loop"))

def show_frames(frames, title="", interval=60, width=3.2):
    """Кадры среды -> анимация в ноутбуке (в превью на GitHub не видна, только при запуске)."""
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w))
    ax.axis("off"); ax.set_title(title, fontsize=10)
    im = ax.imshow(frames[0]); plt.close(fig)
    anim = animation.FuncAnimation(fig, lambda i: (im.set_data(frames[i]),), frames=len(frames), interval=interval, blit=True)
    return HTML(anim.to_jshtml(default_mode="loop"))

## 1. Интерфейс Gymnasium

Любая среда в Gymnasium следует единому интерфейсу:

* `env.reset(seed=...)` -> `(observation, info)` — сбросить среду в начальное состояние;
* `env.step(action)` -> `(observation, reward, terminated, truncated, info)` — сделать шаг;
* `env.observation_space`, `env.action_space` — описание пространств состояний и действий.

`terminated` — эпизод закончился естественным образом (дошли до цели или упали в прорубь), `truncated` — эпизод прерван искусственно (по лимиту шагов, здесь 100).

Начнём с **Frozen Lake** из лекции: агент ходит по замёрзшему озеру 4×4 от старта `S` к цели `G`, в клетках `H` — проруби. Награда 1 только за достижение цели. Состояние — номер клетки от 0 до 15 (по строкам, слева направо), действия: `0` — влево, `1` — вниз, `2` — вправо, `3` — вверх.

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
desc = lake_desc(env)
n_states, n_actions = env.observation_space.n, env.action_space.n
print("observation_space:", env.observation_space, " action_space:", env.action_space)

obs, info = env.reset(seed=0)
print("начальное состояние:", obs)
for step in range(5):
    action = env.action_space.sample()                       # случайное действие
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step={step} action={ARROWS[action]} obs={obs:2d} reward={reward} terminated={terminated}")
    if terminated or truncated:
        obs, info = env.reset()

print("\nномера состояний:"); print(np.arange(16).reshape(4, 4))
draw_lake(desc); plt.show()

## 2. Политика как таблица

В лекции политика — таблица `n_states × n_actions` с вероятностями действий. Детерминированный маршрут — частный случай: в каждой строке одна единица. Напишем маршрут руками как словарь «клетка → действие» и превратим его в таблицу; клетки, которых нет в словаре, получают действие по умолчанию.

Функции `discounted_return` и `run_session` — те же, что в лекции.

In [ ]:
GAMMA = 0.95

def discounted_return(rewards, gamma=GAMMA):
    """G = r_0 + γ r_1 + γ² r_2 + ..."""
    return sum(gamma ** t * r for t, r in enumerate(rewards))

def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей policy. Возвращает состояния (включая последнее), действия и награды."""
    s, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, rewards = [s], [], []
    for _ in range(max_steps):
        a = int(rng.choice(len(policy[s]), p=policy[s]))      # действие ~ π(· | s)
        s, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        if terminated or truncated:
            break
    return states, actions, rewards


LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

def table_policy(route, n_states=16, n_actions=4, default=LEFT):
    """Словарь {клетка: действие} -> таблица вероятностей (детерминированная политика)."""
    policy = np.zeros((n_states, n_actions))
    for s in range(n_states):
        policy[s, route.get(s, default)] = 1.0
    return policy

def success_rate(env, policy, n_episodes=500, seed=0):
    """Доля эпизодов, дошедших до цели."""
    rng = np.random.default_rng(seed)
    return np.mean([sum(run_session(env, policy, rng)[2]) > 0 for _ in range(n_episodes)])

route = {0: DOWN, 4: DOWN, 8: RIGHT, 9: DOWN, 13: RIGHT, 14: RIGHT}
policy_hand = table_policy(route)

rng = np.random.default_rng(0)
states, actions, rewards = run_session(env, policy_hand, rng)
print("маршрут:", states, " return =", round(discounted_return(rewards), 3))
draw_policy(policy_hand, desc, title="политика-маршрут"); plt.show()

uniform_policy = np.ones((n_states, n_actions)) / n_actions
for slippery in [False, True]:
    e = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=slippery)
    print(f"is_slippery={slippery!s:5}: маршрут {success_rate(e, policy_hand):5.1%}, "
          f"равномерная политика {success_rate(e, uniform_policy):5.1%}")

На скользком льду агент идёт туда, куда хотел, с вероятностью 1/3, иначе его сносит вбок (в лекции мы подсмотрели это в `env.unwrapped.P`). Маршрут, идеальный на гладком льду, доходит до цели в единицах процентов случаев: его сносит в проруби, а в клетках, куда агент «не собирался», таблица делает шаг по умолчанию.

**TODO 2.1.** Заполните таблицу для всех 16 клеток так, чтобы на скользком льду доля успехов была не меньше 30%. Подсказки:

* шаг в стену оставляет на месте, а снос работает перпендикулярно выбранному направлению — поэтому у края озера выгодно «идти в стену»: из клетки 0 действие ← даёт исходы «остался в 0», «в 0» (снос вверх — тоже стена) и «в 4», и ни один не ведёт в прорубь;
* рядом с прорубью выбирайте действие, при котором **ни один из трёх исходов** не ведёт в неё, даже если оно не приближает к цели;
* лучшая детерминированная таблица даёт около 74% — её мы научимся находить автоматически на неделе 3.

In [ ]:
route_slippery = dict(route)          # начните с маршрута для гладкого льда
# TODO: дополните и исправьте словарь для всех 16 клеток
policy_slippery = table_policy(route_slippery)

slip = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
rate = success_rate(slip, policy_slippery, n_episodes=2000)
print(f"доля успехов на скользком льду: {rate:.1%}")
draw_policy(policy_slippery, desc, title="ваша таблица для скользкого льда"); plt.show()
assert rate >= 0.3, "маршрут пока ненадёжный, попробуйте ещё"

## 3. Оценка политики и ценность клетки

Качество политики — ожидаемый return $J(\pi) = \mathbb{E}_\pi[G]$; оценка по Монте-Карло — сыграть $K$ эпизодов и усреднить. Мы только что делали это в `success_rate`, но там усреднялся факт успеха, а не return. Посчитаем return.

In [ ]:
def evaluate(env, policy, n_episodes=1000, seed=0):
    """Оценка J(π) по Монте-Карло: средний return и его стандартная ошибка."""
    rng = np.random.default_rng(seed)
    returns = np.array([discounted_return(run_session(env, policy, rng)[2]) for _ in range(n_episodes)])
    return returns.mean(), returns.std() / np.sqrt(n_episodes)

for name, policy in [("равномерная", uniform_policy), ("маршрут", policy_hand)]:
    mean, err = evaluate(env, policy)
    print(f"{name:12s}: J(π) ≈ {mean:.3f} ± {err:.3f}")

Маршрут даёт ровно $0.95^5 \approx 0.774$: шесть шагов до цели, награда 1 на последнем.

Теперь — **ценность клетки** из лекции: средний return эпизодов, считая от момента **первого** попадания в клетку $s$:

$$
V^\pi(s) \approx \frac{1}{N(s)} \sum_{\text{эпизоды, где был } s} G_{t_s}, \qquad t_s - \text{шаг первого попадания в } s .
$$

**TODO 3.1.** Реализуйте `estimate_values`. Для каждого эпизода пройдите по состояниям `states[:-1]` (последнее состояние терминальное, на нём действия уже не было); если состояние встретилось в этом эпизоде впервые, добавьте `discounted_return(rewards[t:])` в его сумму. Клетки, куда ни один эпизод не заходил, должны получить `np.nan`.

In [ ]:
def estimate_values(sessions, n_states, gamma=GAMMA):
    total, count = np.zeros(n_states), np.zeros(n_states)
    # TODO: ваш код здесь
    raise NotImplementedError
    return np.divide(total, count, out=np.full(n_states, np.nan), where=count > 0)

# Проверка на двух игрушечных эпизодах: состояния / действия / награды.
toy = [([0, 1, 3], [RIGHT, RIGHT], [0.0, 1.0]),
       ([0, 0, 1, 3], [UP, RIGHT, RIGHT], [0.0, 0.0, 1.0])]
V_toy = estimate_values(toy, 4)
assert np.isclose(V_toy[0], (0.95 + 0.95 ** 2) / 2), "для клетки 0 нужно среднее по двум эпизодам от ПЕРВОГО попадания"
assert np.isclose(V_toy[1], 1.0) and np.isnan(V_toy[2]) and np.isnan(V_toy[3])
print("ok")

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
for ax, (name, policy) in zip(axes, [("равномерная политика", uniform_policy), ("маршрут", policy_hand)]):
    sessions = [run_session(env, policy, rng) for _ in range(3000)]
    plot_values(estimate_values(sessions, n_states), desc, ax=ax, title=f"V(s): {name}", vmax=1)
plt.show()

Вопросы для обсуждения:

* Почему у маршрута ценности вдоль пути — это степени $0.95$, а в остальных клетках прочерки?
* Почему у равномерной политики ценность старта почти ноль, хотя цель достижима за 6 шагов?
* Ценность цели и прорубей на карте не рисуется. Чему она равна по определению и почему?

## 4. Метод Cross-Entropy по шагам

Три шага из лекции: **сыграть** $K$ эпизодов, **отобрать элиту** (эпизоды не хуже квантиля уровня $q$ **и строго лучше самого плохого**), **подстроить политику** (частоты действий элиты, со сглаживанием по Лапласу $\lambda$ и смешиванием со старой политикой с весом $\alpha$). Шаг 1 у нас уже есть — это `run_session`. Шаги 2 и 3 напишите сами; проверки — на игрушечных данных, где ответ считается руками.

**TODO 4.1.** `select_elite(sessions, scores, q)` — вернуть список эпизодов с `scores[i] >= quantile(scores, q)` и `scores[i] > min(scores)`.

In [ ]:
def select_elite(sessions, scores, q=0.7):
    # TODO: ваш код здесь
    raise NotImplementedError

# Игрушечный пример: 4 эпизода в среде с двумя состояниями и двумя действиями, returns 0, 1, 0, 1.
toy_sessions = [([0, 1, 1], [0, 0], [0.0, 0.0]),
                ([0, 1, 1], [1, 1], [0.0, 1.0]),
                ([0, 0, 0], [0, 1], [0.0, 0.0]),
                ([0, 1, 1], [1, 1], [0.0, 1.0])]
toy_returns = [discounted_return(r) for *_, r in toy_sessions]
elite = select_elite(toy_sessions, toy_returns, q=0.5)
assert elite == [toy_sessions[1], toy_sessions[3]], "в элите должны быть эпизоды 2 и 4 (нумерация с единицы)"
assert select_elite(toy_sessions, [0, 0, 0, 0], q=0.5) == [], "если все эпизоды одинаковы, элита пуста"
print("ok")

**TODO 4.2.** `update_policy(policy, elite, laplace, mix)`:

1. `counts[s, a]` — сколько раз в элите в состоянии `s` сделали `a`, плюс `laplace` в каждой клетке;
2. строки, куда элита заходила (сумма счётчиков больше нуля), нормируем на их сумму, остальные оставляем как в `policy`;
3. возвращаем `mix * new_policy + (1 - mix) * policy`.

Разберите проверку руками: в элите дважды сделали действие 1 в состоянии 0 и дважды — в состоянии 1.

In [ ]:
def update_policy(policy, elite, laplace=0.0, mix=1.0):
    # TODO: ваш код здесь
    raise NotImplementedError

toy_policy = np.full((2, 2), 0.5)
assert np.allclose(update_policy(toy_policy, elite), [[0, 1], [0, 1]])
assert np.allclose(update_policy(toy_policy, elite, laplace=1.0), [[0.25, 0.75], [0.25, 0.75]])
assert np.allclose(update_policy(toy_policy, elite, mix=0.5), [[0.25, 0.75], [0.25, 0.75]])
assert np.allclose(update_policy(toy_policy, [], mix=1.0), toy_policy), "пустая элита не должна менять политику"
print("ok")

Цикл — тот же, что в лекции: играем, отбираем, подстраиваем, записываем историю.

In [ ]:
def cross_entropy_method(env, n_iter=15, n_sessions=200, q=0.7, laplace=0.0, mix=1.0, seed=0):
    rng = np.random.default_rng(seed)
    policy = np.ones((env.observation_space.n, env.action_space.n)) / env.action_space.n
    log = {"policies": [], "success": [], "return": []}
    for _ in range(n_iter):
        sessions = [run_session(env, policy, rng) for _ in range(n_sessions)]     # 1. сыграть
        returns = [discounted_return(r) for *_, r in sessions]
        elite = select_elite(sessions, returns, q)                                # 2. отобрать элиту
        log["policies"].append(policy); log["return"].append(np.mean(returns))
        log["success"].append(np.mean([G > 0 for G in returns]))
        policy = update_policy(policy, elite, laplace, mix)                       # 3. подстроить политику
    return policy, log


policy_cem, log = cross_entropy_method(env, n_iter=15, n_sessions=200, q=0.7, laplace=0.5, mix=0.5)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
axes[0].plot(log["success"], marker="o"); axes[0].set_xlabel("итерация"); axes[0].set_ylabel("доля успешных эпизодов")
draw_policy(policy_cem, desc, ax=axes[1], title="выученная политика"); plt.show()

rate = success_rate(env, policy_cem)
print(f"доля успехов выученной политики: {rate:.1%}")
assert rate >= 0.9

Сравните выученную таблицу с маршрутом из раздела 2 — совпал ли путь? Стрелки показывают лишь самое вероятное действие; в клетках с точкой политика осталась равномерной — элита туда не заходила.

## 5. Эксперименты

### 5.1 Схлопывание и сглаживание

Без сглаживания частоты — очень жёсткое обновление: действие, не попавшее в элиту, получает вероятность ровно 0 и больше никогда не будет опробовано.

In [ ]:
for label, cure in [("без сглаживания", {}), ("Лаплас λ=0.5, смесь α=0.5", dict(laplace=0.5, mix=0.5))]:
    pol, lg = cross_entropy_method(env, n_iter=5, n_sessions=200, **cure)
    print(f"{label:26s}: нулевых вероятностей {(pol == 0).mean():4.0%}, кривая успехов {np.round(lg['success'], 2)}")

### 5.2 Скользкий лёд

Запустим тот же алгоритм на скользком озере и сравним с таблицей, которую вы написали руками в 2.1.

In [ ]:
policy_slip_cem, log_slip = cross_entropy_method(slip, n_iter=30, n_sessions=300, q=0.7, laplace=0.5, mix=0.5)
plt.plot(log_slip["success"], marker="o"); plt.xlabel("итерация"); plt.ylabel("доля успешных эпизодов")
plt.title("Cross-Entropy на скользком льду"); plt.show()
print(f"CEM на скользком льду: {success_rate(slip, policy_slip_cem, n_episodes=2000):.1%}")
print(f"ваша таблица из 2.1:   {success_rate(slip, policy_slippery, n_episodes=2000):.1%}")

Вопросы для обсуждения:

* Почему на скользком льду элитные эпизоды «обманывают»? Что именно попадает в элиту — умные маршруты или везучие?
* Что можно изменить в алгоритме: больше эпизодов на итерацию, выше `q`, меньше `mix`? Попробуйте вживую. Честный ответ: табличный CEM здесь сильно недотягивает до 74%, и в домашнем задании вы разберётесь, почему и что с этим делать.
* Как влияют `q` и `n_sessions` на гладком льду? Попробуйте `q=0.9` и `n_sessions=20`.

### 5.3 Поле 8×8

Домашнее задание — то же самое на карте 8×8: 64 состояния, случайная политика доходит до цели реже, чем в одном эпизоде из пятисот. Посмотрите на карту и прикиньте, сколько эпизодов на итерацию понадобится, чтобы в элиту стабильно попадал хотя бы один успешный.

In [ ]:
env8 = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
draw_lake(lake_desc(env8), size=4); plt.show()
print(f"равномерная политика на 8×8: доля успехов {success_rate(env8, np.ones((64, 4)) / 4, n_episodes=3000):.2%}")

## 6. Что дальше

* **Домашнее задание** (`../homework/homework.ipynb`): Cross-Entropy на Frozen Lake 8×8 — реализация из сегодняшних функций, сглаживание, скользкий лёд; оценка политики и ценность состояний по Монте-Карло; теория (return, марковская цепь, MDP на бумаге). Бонусы: многорукие бандиты и DL-разминка.
* **Если вы не работали с нейросетями**: `dl_basics.ipynb` — нейросети с нуля (модель, функция потерь, градиентный спуск, первая сеть на PyTorch, классификация с кросс-энтропией), затем `pytorch_intro.ipynb` — механика PyTorch для RL. К неделе 5, когда нейросети появятся в лекциях, этого хватит.
* **Неделя 2**: агент, среда, награда и политика подробнее; строим собственную среду в Gymnasium.